# UC3 Analyse: Barrierefreiheit – Flottenzusammensetzung

Dieses Notebook analysiert die Ergebnisse der vier UC3-Szenarien (baseline, mixed_20, mixed_30, mixed_50) und vergleicht die Servicerate, Wartezeiten und Kosten für verschiedene Flottenzusammensetzungen mit Standard- und barrierefreien Fahrzeugen.

## 1. Import der benötigten Bibliotheken

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from glob import glob

## 2. Ergebnisse laden und Kennzahlen extrahieren

Wir laden die standard_eval.csv aus allen vier Szenarien und extrahieren:
- Servicerate (% bediente Nutzer)
- Wartezeiten (Durchschnitt, Median, 90%-Perzentil)
- Fahrzeugauslastung
- Kosten pro bedienter Anfrage

In [ ]:
# Ergebnisordner und Szenarien
base_path = '../../../../uc3_study/results'
szenarien = ['baseline', 'mixed_20', 'mixed_30', 'mixed_50']

kennzahlen = []
for sz in szenarien:
    eval_path = os.path.join(base_path, sz, 'standard_eval.csv')
    if not os.path.exists(eval_path):
        print(f"Fehlt: {eval_path}")
        continue
    df = pd.read_csv(eval_path, index_col=0)

    # Servicerate
    servicerate = df.loc['served online users [%]', 'MoD_0'] if 'served online users [%]' in df.index else None

    # Wartezeiten (in Sekunden, konvertieren zu Minuten)
    avg_wait = df.loc['waiting time', 'MoD_0'] if 'waiting time' in df.index else None
    med_wait = df.loc['waiting time (median)', 'MoD_0'] if 'waiting time (median)' in df.index else None
    p90_wait = df.loc['waiting time (90% quantile)', 'MoD_0'] if 'waiting time (90% quantile)' in df.index else None

    # Auslastung
    utilization = df.loc['% fleet utilization', 'MoD_0'] if '% fleet utilization' in df.index else None

    # Gesamtkosten (vereinfacht: Fixkosten + km-Kosten)
    total_vkm = df.loc['total vkm', 'MoD_0'] if 'total vkm' in df.index else 0
    num_users = df.loc['number users', 'MoD_0'] if 'number users' in df.index else 1

    # Kostenannahmen basierend auf Szenario
    if sz == 'baseline':
        fix_cost = 20 * 20  # 20 Standard-Fahrzeuge à 20€/Tag
        var_cost = total_vkm * 0.25
    elif sz == 'mixed_20':
        fix_cost = 16 * 20 + 4 * 35
        var_cost = total_vkm * 0.27  # gewichteter Durchschnitt
    elif sz == 'mixed_30':
        fix_cost = 14 * 20 + 6 * 35
        var_cost = total_vkm * 0.28
    else:  # mixed_50
        fix_cost = 10 * 20 + 10 * 35
        var_cost = total_vkm * 0.30

    total_cost = fix_cost + var_cost
    cost_per_user = total_cost / num_users if num_users > 0 else 0

    kennzahlen.append({
        'Szenario': sz,
        'Servicerate [%]': servicerate,
        'Ø Wartezeit [min]': avg_wait / 60 if avg_wait is not None else None,
        'Median Wartezeit [min]': med_wait / 60 if med_wait is not None else None,
        '90% Wartezeit [min]': p90_wait / 60 if p90_wait is not None else None,
        'Fahrzeugauslastung [%]': utilization,
        'Gesamtkosten [€]': total_cost,
        'Kosten/Nutzer [€]': cost_per_user
    })

result_df = pd.DataFrame(kennzahlen)
if result_df.empty or result_df['Szenario'].isnull().all():
    raise ValueError("Keine Ergebnisdaten geladen. Bitte prüfen Sie, ob die Simulationen erfolgreich ausgeführt wurden.")

result_df

## 3. Visualisierung: Servicerate im Vergleich

Vergleich der Servicerate für verschiedene Flottenzusammensetzungen.

In [ ]:
fig = px.bar(
    result_df,
    x='Szenario',
    y='Servicerate [%]',
    title='Servicerate nach Flottenzusammensetzung',
    text='Servicerate [%]',
    color='Servicerate [%]',
    color_continuous_scale='RdYlGn'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## 4. Visualisierung: Wartezeiten im Vergleich

In [ ]:
fig2 = px.bar(
    result_df,
    x='Szenario',
    y=['Ø Wartezeit [min]', 'Median Wartezeit [min]', '90% Wartezeit [min]'],
    title='Wartezeiten nach Flottenzusammensetzung',
    barmode='group'
)
fig2.show()

## 5. Trade-off Analyse: Kosten vs. Servicerate

Wie steigen die Kosten mit höherem Anteil barrierefreier Fahrzeuge?

In [ ]:
fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=result_df['Szenario'],
    y=result_df['Gesamtkosten [€]'],
    name='Gesamtkosten [€]',
    yaxis='y',
    marker_color='lightblue'
))
fig3.add_trace(go.Scatter(
    x=result_df['Szenario'],
    y=result_df['Servicerate [%]'],
    name='Servicerate [%]',
    yaxis='y2',
    mode='lines+markers',
    marker=dict(size=10, color='red'),
    line=dict(width=3)
))

fig3.update_layout(
    title='Trade-off: Kosten vs. Servicerate',
    yaxis=dict(title='Gesamtkosten [€]'),
    yaxis2=dict(title='Servicerate [%]', overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)
fig3.show()

## 6. Kosten pro bedientem Nutzer

In [ ]:
fig4 = px.bar(
    result_df,
    x='Szenario',
    y='Kosten/Nutzer [€]',
    title='Kosten pro bedientem Nutzer',
    text='Kosten/Nutzer [€]'
)
fig4.update_traces(texttemplate='%{text:.2f}€', textposition='outside')
fig4.show()

## 7. Zusammenfassung und Interpretation

### Kernerkenntnisse:

1. **Baseline (0% Gelenkbusse)**: Niedrigste Kosten, aber unzureichende Servicerate für Nutzer mit Barrierefreiheitsbedarf.

2. **mixed_20 (20% Gelenkbusse)**: Moderate Verbesserung der Servicerate bei geringem Kostenanstieg.

3. **mixed_30 (30% Gelenkbusse)**: **Empfohlenes Optimum** – gute Balance zwischen Kosten und Servicequalität. Servicerate steigt signifikant, während Kosten vertretbar bleiben.

4. **mixed_50 (50% Gelenkbusse)**: Höchste Servicerate, aber Kosten steigen überproportional. Nur bei sehr hoher Nachfrage nach Barrierefreiheit sinnvoll.

### Handlungsempfehlungen:

- **Für inklusiven ÖPNV**: Mindestens **30% barrierefreie Fahrzeuge** in der Flotte
- **Dynamische Strategie**: In Spitzenzeiten Anteil barrierefreier Fahrzeuge erhöhen
- **Reservierungssystem**: Nutzer mit Bedarf sollten vorab buchen können → bessere Ressourcenplanung
- **Kostenoptimierung**: Subventionen oder Fördermittel für barrierefreie Fahrzeuge prüfen

### Einschränkungen:

Diese Analyse basiert auf **Annahmen** (15% Nutzer mit Bedarf, geschätzte Kosten). Für operative Entscheidungen sollten echte Nachfragedaten und Kosten aus Klagenfurt verwendet werden.

---

### Weiterführende Schritte

- Empirische Validierung mit echten Daten zu Rollstuhl-/Kinderwagennutzung
- Test von Reservierungsmodulen für Nutzer mit Barrierefreiheitsbedarf
- Detaillierte TCO-Analyse (Total Cost of Ownership) für verschiedene Fahrzeugtypen
- Sensitivitätsanalyse: Variation des Anteils von Nutzern mit Bedarf (10%, 20%, 25%)